In [25]:
%%capture
%load_ext sql
%sql duckdb://
%sqlcmd explore --table '/tmp/warehouse_oa/*.parquet'
%config SqlMagic.named_parameters="enabled"
%config SqlMagic.displaylimit = None

In [18]:
%%sql
SELECT
        track_id,
        MIN(CASE WHEN event_name = 'enter_zone_entrance' THEN event_ts END) as start_ts,
        MAX(CASE WHEN event_name IN ('enter_zone_exit', 'leave_zone_exit') THEN event_ts END) as end_ts
    FROM '/tmp/warehouse_oa/*.parquet'
    GROUP BY track_id
    HAVING MIN(CASE WHEN event_name = 'enter_zone_entrance' THEN event_ts END) IS NOT NULL
    AND MAX(CASE WHEN event_name IN ('enter_zone_exit', 'leave_zone_exit') THEN event_ts END) IS NOT NULL
    AND MIN(CASE WHEN event_name = 'enter_zone_entrance' THEN event_ts END) <
        MAX(CASE WHEN event_name IN ('enter_zone_exit', 'leave_zone_exit') THEN event_ts END) ORDER BY start_ts DESC LIMIT 15

Running query in 'duckdb://'

track_id,start_ts,end_ts
b'\x02\xf3\x96\x90\xf7(E\x13\xb0e\x80\x07n\xfd)M',2025-01-07 22:11:20.315487,2025-01-07 22:11:36.512343
b'&1\xd9\x11\xd8DH\xd5\xa8\xae\x05xb|Ga',2025-01-07 22:08:34.370515,2025-01-07 22:08:49.040785
b'A[\xc4\x03\xa1gJ\xc4\x86\xd6\xeal\x9b\xd0e\xbf',2025-01-07 22:07:00.749475,2025-01-07 22:07:13.162680
b'\xadj\xeb\x81m\xe3@\x82\xa9\x85\x91\x93Ya\xb2|',2025-01-07 22:06:59.613189,2025-01-07 22:07:12.186551
"b""\xe7\xa3'\xb6\xed\x9cB\r\x8f\x92\xc2\xb24\x86\xc3$""",2025-01-07 22:05:33.327715,2025-01-07 22:05:52.926903
b'`\x92\r\xde.\xe9Dc\xbd7\xbf\x81v\xd5\xd4\xca',2025-01-07 22:05:29.887371,2025-01-07 22:05:53.761244
b'Htl\xf0M9K\n\xba\x81\xa3\x89\x92>\xb7l',2025-01-07 22:05:26.623801,2025-01-07 22:05:48.741341
b'\xf7g?\xf7\x14\xf0L=\x88\x19\x01\xb5Eu\xd7\xe5',2025-01-07 22:04:57.911483,2025-01-07 22:05:11.623361
b'\xd1\xef\xf3\n\x17\xb3Og\xa5\x86\xf3\xf2\x16uo\xb3',2025-01-07 22:04:56.439322,2025-01-07 22:05:11.298433
"b'_?E}""vC\x1d\x93/\xfba\x84\xb5\xc2\xbf'",2025-01-07 22:04:35.124843,2025-01-07 22:04:42.404023


In [ ]:
%%sql
SELECT *, hex(track_id) as track_id_hex from '/tmp/warehouse_oa/*.parquet' WHERE track_id_hex GLOB '50A2C2BC*'

Running query in 'duckdb://'

event_ts,event_id,event_name,track_conf,track_class,track_id,cam_id,track_id_hex
2025-01-07 21:46:17.603335,b'\x11\xe6#w\x895I\x1a\x80\xd1MHa\xc2\x06\xf7',enter_zone_center,74,0,b'P\xa2\xc2\xbc\rEL\xc0\x82\xd6_\xdb9\x96\xaf\x96',b'.\xfd\xad\x08-\xa0D\xbd\x8bb)\x00\x9b\xc2S\x89',50A2C2BC0D454CC082D65FDB3996AF96


In [ ]:
%%sql --save base_events --no-execute
SELECT 
  track_id,
  event_ts,
  DATE_TRUNC('minute', event_ts) - 
    (INTERVAL '1 minute' * (EXTRACT(MINUTE FROM event_ts)::int % 5)) as bucket_5min,
  event_name,
  track_conf,
  REGEXP_REPLACE(event_name, '^(enter|leave)_zone_', '') as zone_name,
  CASE 
    WHEN event_name LIKE 'enter%' THEN 'enter'
    WHEN event_name LIKE 'leave%' THEN 'leave'
  END as action_type
FROM '/tmp/warehouse_oa/*.parquet'
WHERE event_name LIKE '%zone_%'


Running query in 'duckdb://'

Skipping execution...

In [ ]:
%%sql --with base_events --save zone_pairs --no-execute
  SELECT 
    e.track_id,
    e.zone_name,
    e.event_ts as enter_time,
    MIN(l.event_ts) as leave_time,
    e.track_conf,
    e.bucket_5min as time_bucket
  FROM base_events e
  LEFT JOIN base_events l ON 
    e.track_id = l.track_id AND
    e.zone_name = l.zone_name AND
    l.action_type = 'leave' AND
    l.event_ts > e.event_ts
  WHERE e.action_type = 'enter'
  GROUP BY 
    e.track_id,
    e.zone_name,
    e.event_ts,
    e.track_conf,
    e.bucket_5min

Running query in 'duckdb://'

Skipping execution...

In [ ]:
%%sql --with zone_pairs --save visit_durations --no-execute
SELECT
    track_id,
    zone_name,
    time_bucket,
    enter_time,
    leave_time,
    EXTRACT(EPOCH FROM (leave_time - enter_time)) as duration_seconds,
    track_conf
  FROM zone_pairs
  WHERE leave_time IS NOT NULL

Running query in 'duckdb://'

Skipping execution...

In [21]:
%%sql --with visit_durations --save bucket_data
SELECT 
  time_bucket,
  zone_name,
  COUNT(*) as total_visits,
  ROUND(AVG(duration_seconds), 2) as avg_duration_seconds,
  ROUND(MIN(duration_seconds), 2) as min_duration_seconds,
  ROUND(MAX(duration_seconds), 2) as max_duration_seconds,
  ROUND(AVG(track_conf), 2) as avg_track_confidence,
  COUNT(DISTINCT track_id) as unique_tracks
FROM visit_durations
GROUP BY 
  time_bucket,
  zone_name
ORDER BY 
  time_bucket,
  zone_name;

Running query in 'duckdb://'

time_bucket,zone_name,total_visits,avg_duration_seconds,min_duration_seconds,max_duration_seconds,avg_track_confidence,unique_tracks
2025-01-07 21:35:00,center,12,7.09,1.29,23.64,87.17,11
2025-01-07 21:35:00,entrance,9,33.27,0.48,141.87,83.56,7
2025-01-07 21:35:00,exit,5,16.73,1.32,70.9,85.8,5
2025-01-07 21:40:00,center,18,21.95,2.43,218.1,84.0,17
2025-01-07 21:40:00,entrance,7,23.08,0.16,115.7,80.29,7
2025-01-07 21:40:00,exit,4,3.77,2.44,4.69,79.75,4
2025-01-07 21:45:00,center,15,28.08,1.62,180.88,87.0,15
2025-01-07 21:45:00,entrance,10,8.33,0.65,36.98,79.1,10
2025-01-07 21:45:00,exit,7,3.14,0.81,6.12,86.57,7
2025-01-07 21:50:00,center,13,9.57,1.46,74.47,88.46,12


In [24]:
%%sql
SELECT * FROM base_events

Generating CTE with stored snippets: 'base_events'

Running query in 'duckdb://'

track_id,event_ts,bucket_5min,event_name,track_conf,zone_name,action_type
b'\x8d\x8cGx\xca\xbdJ:\xa4x\xee\xce)~\x0cU',2025-01-07 22:11:54.927773,2025-01-07 22:10:00,leave_zone_center,83,center,leave
b'\x8d\x8cGx\xca\xbdJ:\xa4x\xee\xce)~\x0cU',2025-01-07 22:11:55.094263,2025-01-07 22:10:00,enter_zone_entrance,89,entrance,enter
b'\xc51\xcal\xa5KE\xdc\x9a\xabX1\xdb\xf6\xc9\x01',2025-01-07 22:11:55.751636,2025-01-07 22:10:00,leave_zone_center,78,center,leave
b'\xc51\xcal\xa5KE\xdc\x9a\xabX1\xdb\xf6\xc9\x01',2025-01-07 22:11:55.917001,2025-01-07 22:10:00,enter_zone_entrance,85,entrance,enter
b'\xc51\xcal\xa5KE\xdc\x9a\xabX1\xdb\xf6\xc9\x01',2025-01-07 22:11:54.108696,2025-01-07 22:10:00,enter_zone_center,78,center,enter
b'\x8d\x8cGx\xca\xbdJ:\xa4x\xee\xce)~\x0cU',2025-01-07 22:11:52.149159,2025-01-07 22:10:00,enter_zone_center,83,center,enter
b'.\xf7\\\x8b}*C\xcd\xab\xb8\xb8\xa6\x88\r\xa2)',2025-01-07 22:11:48.560269,2025-01-07 22:10:00,leave_zone_center,86,center,leave
b'.\xf7\\\x8b}*C\xcd\xab\xb8\xb8\xa6\x88\r\xa2)',2025-01-07 22:11:48.722352,2025-01-07 22:10:00,enter_zone_entrance,89,entrance,enter
b'm\x99\xdb\xd0\xce\x89BQ\x87\xd2\x17\xd5(\xa97\x06',2025-01-07 22:11:48.722352,2025-01-07 22:10:00,enter_zone_entrance,81,entrance,enter
b'.\xf7\\\x8b}*C\xcd\xab\xb8\xb8\xa6\x88\r\xa2)',2025-01-07 22:11:47.252702,2025-01-07 22:10:00,leave_zone_exit,84,exit,leave
